In [1]:
#setup
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'

In [ ]:
import bs4
from langchain_community.document_loaders import PyPDFDirectoryLoader

directory_path = "RAG_project"
loader = PyPDFDirectoryLoader(directory_path)
med_docs = loader.load()


Loaded 104 pages across all PDFs.


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(med_docs)

In [4]:
#index

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=OpenAIEmbeddings())

retriever = vectorstore.as_retriever()

In [13]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=OpenAIEmbeddings())


retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [16]:
docs = retriever.invoke("What do imaging results show for those with ATFL injuries?")

In [9]:
len(docs)
print(docs[0].page_content)
print(docs[0].metadata)

traumatic arthritis, bone necrosis[54], and other complications, which seriously affect the quality of life 
and sports activities. It is difficult to predict the time required for natural ligament healing after acute 
ATFL injury.  Therefore,  it is paramount  to judge  injury  degree  accurately  and select  appropriate 
treatment measures accordingly. After comparing the various factors influencing rehabilitation results 
and economic costs of conservative and surgical treatment, it was found that conservative treatment is 
not inferior to surgical treatment[55]; in fact, patients have higher satisfaction and can resume pre-injury 
work and sports activities earlier.
Surgical treatment
By understanding  the etiology  and pathogenesis  of acute  ATFL  injury  and implementing  early 
diagnosis and treatment, the occurrence of complications can be effectively prevented. About 80%-85% 
of type I, type II, and some severe ATFL acute sprains can be cured through a rehabilitation treatmen

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


# Prompt
template = """You are an assistant for question-answering tasks related to ATFL injuries.

Use the following retrieved context to answer the question.
If you don't know the answer based on the context, say that you don't know.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(model="gpt-6-luna")


In [19]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is the recovery process like for ankle injuries?")

'Recovery is generally gradual and guided by the athlete’s symptoms, rehabilitation progress, and goals. The current approach tends to favor earlier movement and weightbearing when appropriate, rather than prolonged immobilization, which can contribute to stiffness, muscle loss, and reduced proprioception. Preventing physical deconditioning should begin early.\n\nReturn to sport comes after the athlete has met the preceding rehabilitation goals and is physically and mentally ready. Practice and playing time are then increased progressively, with movement complexity building over time, until the athlete can resume usual sport confidently. Recovery expectations should be discussed and managed throughout.'